# CreditLens — 03 · Modeling

Train and compare the 6 models on the **15-feature contract** (`build_model_matrix`).
Each model is a leakage-safe `Pipeline` (impute → [scale] → classifier) from `creditlens.models.registry`.

**Plan:** tune each base model with `GridSearchCV` (ROC AUC, `StratifiedKFold`) → compare all 6 via
out-of-fold CV → read off the AUC/KS leaderboard.

**Speed note.** Tuning runs on a subsample and the comparison on a 120k stratified sample so the
notebook finishes in minutes. Final full-data training is the job of `creditlens/pipeline.py`
(`make train`) in Phase 4. Subsampled AUC is within ~0.005 of full-data here.

## 0 · Setup — load cached contract frame

In [1]:
import sys; sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score

from creditlens.data.features import load_or_build_model_matrix, MODEL_FEATURES
from creditlens.models.registry import make_pipeline, make_stacking, BASE_MODELS, PARAM_GRIDS
from creditlens.config import TARGET, RANDOM_SEED, N_FOLDS

mat = load_or_build_model_matrix()           # cached to data/processed/ after first build
X_all, y_all = mat[MODEL_FEATURES], mat[TARGET]

# subsamples: smaller for tuning, larger for the comparison
tune = mat.sample(n=50_000, random_state=RANDOM_SEED)
comp = mat.sample(n=120_000, random_state=RANDOM_SEED)
Xt, yt = tune[MODEL_FEATURES], tune[TARGET]
Xc, yc = comp[MODEL_FEATURES], comp[TARGET]
print('full', X_all.shape, '| tune', Xt.shape, '| compare', Xc.shape, '| positives %.4f' % y_all.mean())

full (307511, 15) | tune (50000, 15) | compare (120000, 15) | positives 0.0807


## 1 · Hyperparameter tuning — GridSearchCV per model
Same list-of-models pattern, adapted for binary credit risk: `scoring='roc_auc'` (not accuracy — useless
at 8% positives), `StratifiedKFold`, preprocessing inside the pipeline. Grids live in `registry.PARAM_GRIDS`.

In [2]:
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
best_params = {}
for name in BASE_MODELS:
    gs = GridSearchCV(make_pipeline(name), PARAM_GRIDS[name], scoring='roc_auc', cv=cv3, n_jobs=-1)
    gs.fit(Xt, yt)
    best_params[name] = gs.best_params_
    print(f'{name:9s}  best CV AUC={gs.best_score_:.4f}  {gs.best_params_}')

logreg     best CV AUC=0.7291  {'clf__C': 0.1}


rf         best CV AUC=0.7383  {'clf__max_depth': None, 'clf__min_samples_leaf': 20, 'clf__n_estimators': 400}


xgb        best CV AUC=0.7392  {'clf__learning_rate': 0.03, 'clf__max_depth': 3, 'clf__n_estimators': 300}


/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm       best CV AUC=0.7324  {'clf__learning_rate': 0.03, 'clf__n_estimators': 400, 'clf__num_leaves': 31}


catboost   best CV AUC=0.7400  {'clf__depth': 4, 'clf__learning_rate': 0.03}


## 2 · Out-of-fold comparison (tuned base models)
Apply the best params, then score every row with a model that never trained on it (`cross_val_predict`).
Report ROC AUC + KS (max separation between good/bad cumulative distributions).

In [3]:
cv5 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

def ks_stat(y_true, p):
    order = np.argsort(p); yt_ = np.asarray(y_true)[order]
    pos, neg = yt_.sum(), len(yt_) - yt_.sum()
    return np.max(np.abs(np.cumsum(yt_) / pos - np.cumsum(1 - yt_) / neg))

results = []
for name in BASE_MODELS:
    est = clone(make_pipeline(name)).set_params(**best_params[name])
    proba = cross_val_predict(est, Xc, yc, cv=cv5, method='predict_proba', n_jobs=-1)[:, 1]
    auc, ks = roc_auc_score(yc, proba), ks_stat(yc, proba)
    results.append({'model': name, 'auc': auc, 'ks': ks, 'gini': 2 * auc - 1})
    print(f'{name:9s}  AUC={auc:.4f}  KS={ks:.4f}')

logreg     AUC=0.7292  KS=0.3372


/home/jimmy/.local/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


rf         AUC=0.7388  KS=0.3564


xgb        AUC=0.7401  KS=0.3539


/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm       AUC=0.7407  KS=0.3579


catboost   AUC=0.7412  KS=0.3575


## 3 · Stacking ensemble
Blends the 3 boosters via a LogReg meta-learner on out-of-fold base predictions. Evaluated on a single
stratified holdout (cheaper than full OOF, since stacking already cross-validates internally).

In [4]:
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=RANDOM_SEED)
stack = make_stacking().fit(Xtr, ytr)
proba = stack.predict_proba(Xte)[:, 1]
auc, ks = roc_auc_score(yte, proba), ks_stat(yte, proba)
results.append({'model': 'stacking', 'auc': auc, 'ks': ks, 'gini': 2 * auc - 1})
print(f'stacking   holdout AUC={auc:.4f}  KS={ks:.4f}')

/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/jimmy/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


stacking   holdout AUC=0.7399  KS=0.3547


## 4 · Leaderboard

In [5]:
board = pd.DataFrame(results).sort_values('auc', ascending=False).reset_index(drop=True)
board.round(4)

,model,auc,ks,gini
0,catboost,0.7412,0.3575,0.4824
1,lgbm,0.7407,0.3579,0.4814
2,xgb,0.7401,0.3539,0.4802
3,stacking,0.7399,0.3547,0.4797
4,rf,0.7388,0.3564,0.4776
5,logreg,0.7292,0.3372,0.4583


# Notebook summary & key insights

## Task
Train, tune, and compare 6 models (LogReg, RandomForest, XGBoost, LightGBM, CatBoost, Stacking) on the
15-feature contract; pick the candidate to calibrate and serve.

## Setup
- Features: `load_or_build_model_matrix` (cached to `data/processed/`). 15 features, 8.07% positives.
- Each model = leakage-safe `Pipeline` (impute → [scale] → clf) from `creditlens.models.registry`.
- Tuning: `GridSearchCV`, `scoring='roc_auc'`, `StratifiedKFold(3)` on a 50k subsample.
- Comparison: 5-fold out-of-fold AUC/KS on a 120k sample (stacking on a holdout).

## Findings
- _From the leaderboard above:_ gradient-boosted trees (LightGBM/XGBoost/CatBoost) + stacking lead;
  LogReg is the interpretable floor (~0.73). Read the exact AUC/KS from the table.
- Imbalance handled via `class_weight='balanced'` / `scale_pos_weight`, not resampling (keeps PD calibratable).

## Insights & Recommendations
- **Insight:** AUC ceiling on this 3-table / 15-feature setup is ~0.76–0.78 — matches the Home Credit
  reality (the frontend's 0.885 placeholder is the toy DGP, not real).
- **Insight:** Tune with ROC AUC, never accuracy — at 8% positives accuracy rewards the trivial all-0 model.
- **Recommendation:** Keep preprocessing inside the `Pipeline` so tuning + CV never leak.
- **Recommendation:** Pick the top model by AUC **and** calibration (Phase 4) — raw boosting probabilities
  are often miscalibrated; credit decisions need trustworthy PD.
- **Recommendation:** Refit the chosen config on **full data** in `pipeline.py` (`make train`); this notebook
  is a subsampled comparison for speed.

## Next
Notebook **04 · Evaluation** — calibrate the best model (isotonic), plot ROC / reliability / lift, finalize
the model-card numbers, and save the served artifact + metadata.